# 03 · Qualidade dos dados e extração base

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/renanbrettas/Projeto-ci-ncia-de-dados---Podas-de-rvore/blob/main/03_qualidade_e_extracao.ipynb)

**Projeto:** Chamados de poda de árvore no Rio de Janeiro &nbsp;·&nbsp; **Parte 3 de 5**

---

### Contexto
Para analisar onde os chamados acontecem, é preciso saber quão bem eles estão localizados (bairro, logradouro, coordenada) e fixar uma extração reprodutível.

### Pergunta
> Quão confiável é a localização dos chamados de poda, e qual unidade espacial usar?

### Dados utilizados
- `adm_central_atendimento_1746.chamado`

### Roteiro
1. Localização e justificativa preenchidas, por ano
2. Campo `reclamacoes`
3. Extração base 2021–2025 (`data/poda_2021_2025.parquet`)

### Principais resultados
- **133.715** chamados de poda em 2021–2025.
- Bairro e logradouro preenchidos em ~98–99% dos chamados.
- A coordenada válida cai de **94,7% (2021) para 59,2% (2025)**, por isso a unidade principal é o **logradouro**.
- `reclamacoes` é zero em ~97% dos chamados; a justificativa é pouco preenchida.

> ℹ️ As saídas deste notebook são as da execução original no Google Colab (dados extraídos do BigQuery
> em novembro de 2026), então os resultados podem ser lidos sem rodar nada. Para reproduzir, rode a
> célula de configuração abaixo (veja o [README](README.md)).

In [ ]:
# Configuração (rode primeiro)
import os
import subprocess
import sys
from pathlib import Path

# No Colab, informe o projeto do Google Cloud que vai pagar a leitura (veja o README):
# os.environ["GCP_PROJECT_ID"] = "seu-projeto-gcp"

RAIZ = Path.cwd()
if not (RAIZ / "src" / "podas").exists():  # aberto direto no Colab: baixa o código do repositório
    RAIZ = Path("/content/analise-podas")
    if not RAIZ.exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/renanbrettas/Projeto-ci-ncia-de-dados---Podas-de-rvore.git", str(RAIZ)], check=True)
sys.path.insert(0, str(RAIZ / "src"))

import pandas as pd
from google.cloud import bigquery

from podas.bq import get_client, rodar
from podas.config import ALVO, SENSIBILIDADE, ARQ_PODA, DATA_DIR, FIG_DIR, OUT_DIR

client = get_client()

## 1. Localização e justificativa preenchidas, por ano

In [ ]:
ALVO = [
    "Poda de árvore em logradouro",
    "Poda de árvore em vias públicas",
    "Árvore - Poda",
]
SENSIBILIDADE = ["Poda e remoção de árvore"]
P = [bigquery.ArrayQueryParameter("alvo", "STRING", ALVO)]

# 1) qualidade da localização e preenchimento da justificativa, por ano
sql1 = """
SELECT
  EXTRACT(YEAR FROM data_inicio) AS ano,
  COUNT(*) AS n,
  COUNTIF(id_bairro IS NOT NULL AND TRIM(id_bairro) NOT IN ('', '0')) AS bairro_ok,
  COUNTIF(id_logradouro IS NOT NULL AND TRIM(id_logradouro) NOT IN ('', '0')) AS logr_ok,
  COUNTIF(latitude BETWEEN -23.08 AND -22.74
          AND longitude BETWEEN -43.80 AND -43.09) AS coord_ok,
  COUNTIF(justificativa_status IS NOT NULL
          AND TRIM(justificativa_status) != '') AS just_preenchida
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@alvo)
  AND data_inicio >= DATETIME '2021-01-01' AND data_inicio < DATETIME '2026-01-01'
GROUP BY ano
ORDER BY ano
"""
loc = rodar(sql1, P)
for c in ["bairro_ok", "logr_ok", "coord_ok", "just_preenchida"]:
    loc[c + "_%"] = (100 * loc[c] / loc["n"]).round(1)
print("=== LOCALIZAÇÃO E JUSTIFICATIVA ===")
print(loc.to_string())

# 2) unidade e reclamações (Comlurb vs Ouvidoria)
sql2 = """
SELECT id_unidade_organizacional AS id_un, nome_unidade_organizacional AS nome_un,
       COUNT(*) AS n, SUM(reclamacoes) AS soma_reclamacoes,
       COUNTIF(reclamacoes > 0) AS com_reclamacao
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@alvo)
  AND data_inicio >= DATETIME '2021-01-01' AND data_inicio < DATETIME '2026-01-01'
GROUP BY 1, 2
ORDER BY n DESC
LIMIT 15
"""
print("\n=== UNIDADE E RECLAMAÇÕES ===")
print(rodar(sql2, P).to_string())

custo estimado: 1.518 GB
=== LOCALIZAÇÃO E JUSTIFICATIVA ===
    ano      n  bairro_ok  logr_ok  coord_ok  just_preenchida  bairro_ok_%  logr_ok_%  coord_ok_%  just_preenchida_%
0  2021  23906      23463    23463     22632             1860         98.1       98.1        94.7                7.8
1  2022  25223      25008    25008     21823             2196         99.1       99.1        86.5                8.7
2  2023  31176      31091    31091     20435             5289         99.7       99.7        65.5               17.0
3  2024  27518      26847    26847     16462             6338         97.6       97.6        59.8               23.0
4  2025  25892      25465    25465     15323             6347         98.4       98.4        59.2               24.5

=== UNIDADE E RECLAMAÇÕES ===
custo estimado: 2.153 GB
  id_un                                          nome_un       n  soma_reclamacoes  com_reclamacao
0   118  COMLURB - Companhia Municipal de Limpeza Urbana  123268              4155

- `id_bairro` e `id_logradouro` estão preenchidos em ~98–99% dos chamados.
- A **coordenada válida cai** de 94,7% (2021) para 59,2% (2025), então análises por ponto perdem
  cobertura nos anos recentes. Por isso a unidade espacial principal é o **logradouro**.
- A justificativa de status é pouco preenchida (7,8% → 24,5%).

## 2. Campo `reclamacoes`

In [ ]:
P = [bigquery.ArrayQueryParameter("alvo", "STRING", ALVO)]

# distribuição do campo reclamacoes
sql = """
SELECT reclamacoes, id_unidade_organizacional AS id_un, COUNT(*) AS n
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@alvo)
  AND data_inicio >= DATETIME '2021-01-01' AND data_inicio < DATETIME '2026-01-01'
GROUP BY 1, 2
ORDER BY reclamacoes
"""
print(rodar(sql, P).to_string())

custo estimado: 1.360 GB
   reclamacoes id_un       n
0            0   247   10289
1            0   118  119546
2            1   247     155
3            1   118    3365
4            2   247       3
5            2   118     307
6            3   118      34
7            4   118      12
8            5   118       2
9            8   118       2


Cerca de 97% dos chamados têm zero reclamações; o campo tem pouca variação para modelagem.

## 3. Extração base 2021–2025

In [ ]:
sql_extract = """
SELECT id_chamado, id_bairro, id_logradouro, numero_logradouro,
       data_inicio, id_unidade_organizacional, reclamacoes,
       latitude, longitude
FROM `datario.adm_central_atendimento_1746.chamado`
WHERE subtipo IN UNNEST(@alvo)
  AND data_inicio >= DATETIME '2021-01-01' AND data_inicio < DATETIME '2026-01-01'
"""
cfg = bigquery.QueryJobConfig(query_parameters=P, maximum_bytes_billed=20 * 10**9)
poda_df = client.query(sql_extract, job_config=cfg).to_dataframe()
print(poda_df.shape)

poda_df.to_parquet(ARQ_PODA)

(133715, 9)


In [ ]:
print(poda_df.shape)
print(poda_df.dtypes)
print(poda_df.head(10).to_string())
print(poda_df.isna().sum())

(133715, 9)
id_chamado                           object
id_bairro                            object
id_logradouro                        object
numero_logradouro                     Int64
data_inicio                  datetime64[us]
id_unidade_organizacional            object
reclamacoes                           Int64
latitude                            float64
longitude                           float64
dtype: object
  id_chamado id_bairro id_logradouro  numero_logradouro         data_inicio id_unidade_organizacional  reclamacoes   latitude  longitude
0   22408028       101         53561                111 2025-09-16 10:03:20                       118            0        NaN        NaN
1   22386145        64        197327                 35 2025-09-10 12:51:30                       118            0 -22.899567 -43.290125
2   22432020       117        100610                 82 2025-09-22 10:53:43                       118            0 -22.964723 -43.346768
3   22402792        34        

- **133.715** chamados de poda em 2021–2025.
- **1.841** sem bairro e sem logradouro (são os mesmos registros; ver notebook 04).
- **36.990** (27,7%) sem coordenada.

> Uma tentativa anterior de marcar chamados "dependentes da Light" por texto foi descartada:
> a extração não tem coluna de descrição, e a marcação acabou valendo 100% dos registros.

---

⬅️ [02 · Light, unidades responsáveis e dados do COR](02_light_e_cor.ipynb) &nbsp;|&nbsp; [🏠 Início](README.md) &nbsp;|&nbsp; [04 · Análise espacial: bairros e logradouros](04_analise_espacial.ipynb) ➡️